# Neural ODE Architecture Sweep — Kaggle Runner

Runs the Euler and Heat neural ODE architecture sweep scripts from the repo.
Clone this repo into Kaggle, then run this notebook.

**Expected runtime**: ~2-3 hours on T4 GPU

In [ ]:
# Install JAX with CUDA support FIRST, then neural ODE libraries
!pip install -q --upgrade "jax[cuda12]" equinox diffrax optax opinf numpyro

In [ ]:
# Verify GPU is available
import jax
devices = jax.devices()
print(f"JAX version: {jax.__version__}")
print(f"Devices: {devices}")
assert any(d.platform == "gpu" for d in devices), (
    "No GPU detected! Make sure you enabled GPU in "
    "Settings > Accelerator > GPU T4 x2"
)

In [ ]:
%cd /kaggle/working
!rm -rf probabilistic_rom_inference
!git clone --branch tests https://github.com/Anthony50102/probabilistic_rom_inference.git
%cd probabilistic_rom_inference

In [ ]:
import os, subprocess, zipfile

REPO_ROOT = "/kaggle/working/probabilistic_rom_inference"
SWEEP_DIR = os.path.join(REPO_ROOT, "experiments", "neural_ode")
assert os.path.isfile(os.path.join(SWEEP_DIR, "euler_neural_ode.py")), \n
    f"Can't find euler_neural_ode.py in {SWEEP_DIR}"
print(f"Sweep dir: {SWEEP_DIR}")

In [ ]:
print("=" * 70)
print("Running Euler architecture sweep...")
print("=" * 70)
result = subprocess.run(
    ["python", "-u", "euler_neural_ode.py"],
    cwd=SWEEP_DIR,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
print(f"Euler exit code: {result.returncode}")

In [ ]:
print("=" * 70)
print("Running Heat architecture sweep...")
print("=" * 70)
result = subprocess.run(
    ["python", "-u", "heat_neural_ode.py"],
    cwd=SWEEP_DIR,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
print(f"Heat exit code: {result.returncode}")

In [ ]:
FIGURES_DIR = os.path.join(SWEEP_DIR, "figures")
ZIP_PATH = "/kaggle/working/neural_ode_sweep_results.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FIGURES_DIR):
        for f in files:
            filepath = os.path.join(root, f)
            arcname = os.path.relpath(filepath, os.path.dirname(FIGURES_DIR))
            zf.write(filepath, arcname)

n_files = len([f for f in os.listdir(FIGURES_DIR) if f.endswith(".png")])
print(f"
📦 Zipped {n_files} figures to: {ZIP_PATH}")
print("Download from the Output tab on the right →")